# Paper 73 backbone on YOLO26n: SAFA-WT inside the C3k2 at layers 6 and 8

| Part | Source |
|---|---|
| **Backbone (layers 0–10)** | **Paper 73:** a **SAFA-WT** block (Haar wavelet + learnable soft-thresholding + frequency attention) added inside the bottleneck branch of the C3k2 at **layer 6 (P4)** and **layer 8 (P5)**; every other backbone layer is YOLO26n base |
| Neck FPN + PAN (layers 11–22) | YOLO26n base, unchanged (the SAFA-WT after neck fusion in Paper 73 is not used) |
| Head (Detect, layer 23) | YOLO26n base, unchanged (one-to-one, NMS-free; the OBB angle branch of Paper 73 is not used) |

The baseline ("Standard YOLO26n Backbone") is **not trained here**; its values are already filled in the final table.

**Kaggle settings:** Accelerator = **GPU T4 x1**, Internet = **On**, keep the notebook **Private**

## 1. Setup

In [ ]:
# ultralytics version is pinned: use the same version on Kaggle, Raspberry Pi 5 and Jetson
!pip install -q ultralytics==8.4.137 roboflow

In [ ]:
import os, sys, gc, shutil, random
from copy import deepcopy
from pathlib import Path

import numpy as np
import pandas as pd
import torch, torchvision
import ultralytics

print("ultralytics", ultralytics.__version__, "| torch", torch.__version__, "| torchvision", torchvision.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE (turn on the GPU accelerator)")

WORK = Path("/kaggle/working")
os.chdir(WORK)
sys.path.insert(0, str(WORK))  # so safawt_backbone.py can be imported

## 2. Dataset (download from Roboflow)

The Roboflow API key is written in the cell below. Keep this notebook **Private** on Kaggle.

In [ ]:
import os, yaml
from roboflow import Roboflow

os.chdir("/kaggle/working")

# Roboflow API key (keep this notebook Private on Kaggle)
API_KEY = "CxHgc1Nn3FK5vYNsBtJW"

rf = Roboflow(api_key=API_KEY)
project = rf.workspace("md-mahibullah").project("digitalaccess-2-copy")
dataset = project.version(4).download("yolo26")  # Roboflow YOLO26 export (same label format as YOLOv8)

# Rename and fix paths
new = "/kaggle/working/digitalaccess"
if os.path.exists(dataset.location) and not os.path.exists(new):
    os.rename(dataset.location, new)

# Update YAML
yaml_path = f"{new}/data.yaml"
with open(yaml_path) as f:
    data = yaml.safe_load(f)
data["path"], data["train"], data["val"] = new, "train/images", "valid/images"
if os.path.exists(f"{new}/test/images"):
    data["test"] = "test/images"
else:
    data.pop("test", None)
with open(yaml_path, "w") as f:
    yaml.dump(data, f)

# the following cells use these variables
DATA_ROOT = Path(new)
DATA_YAML = yaml_path
names = data["names"]
assert data.get("test"), "No test split: the evaluation and FPS steps need a test split"

for s in ["train", "valid", "test"]:
    print(f"{s:6s}: {len(list((DATA_ROOT / s / 'images').glob('*')))} images")
print("nc =", len(names))
print("✅ Dataset prepared!")

## 3. Paper 73 backbone: SAFA-WT

**What SAFA-WT does** (Paper 73, Section II-B/C):
1. **Haar DWT** splits a feature map into 4 half-size bands: LL (low frequency) and LH, HL, HH (high frequency).
   It is a stride-2 depthwise conv with the 4 fixed Haar filters (frozen), so it is exact and lossless.
2. A **depthwise 3×3 conv** works inside each band. With more than one level, the LL band is split again (cascade).
3. **Learnable soft-thresholding (ST)** on the high bands: `sign(y)·max(|y| − τ, 0)` with one τ per channel, which removes
   small (noisy) responses and keeps strong edges.
4. **Frequency attention (FA)**: GAP over the 4 bands → FC → ReLU → FC → sigmoid gives one weight per band and channel.
5. **IWT** (transposed conv with the same Haar filters) rebuilds the map from the weighted bands, from the deepest level up.

**Where it sits in YOLO26n.** Paper 73 replaces the CSPDarknet bottleneck with SAFA-WT. To keep the pretrained C3k2,
SAFA-WT is added **after** the pretrained C3k block inside the C3k2, as a residual: `out = y + γ · SAFA-WT(y)`.

| Layer | Base YOLO26n | Paper 73 backbone | Wavelet levels | Map size at 640 |
|---|---|---|---|---|
| 6 | C3k2 (P4, inner width 64) | C3k2 with **C3k → SAFA-WT** | 2 | 40 × 40 → 20 → 10 |
| 8 | C3k2 (P5, inner width 128) | C3k2 with **C3k → SAFA-WT** | 1 | 20 × 20 → 10 |

Channels and map sizes stay the same, so no layer index changes and the neck connections stay the same.

**Pretrained weights (identity start).** `yolo26n.pt` is loaded first and every pretrained layer keeps its weights.
γ starts at **0**, so at step 0 the network is exactly YOLO26n base; training then opens the SAFA-WT branch.

**Not taken from Paper 73:** the SAFA-WT after neck fusion (neck stays base), the OBB head (we use axis-aligned boxes),
and the training recipe (AdamW, 300 epochs, no pretraining). Our fixed recipe is used, so the comparison stays fair.

**NCNN note.** Everything is plain conv / transposed conv / pooling / sigmoid / max / min. Soft-thresholding is written as
`y − min(max(y, −τ), τ)`, which is the same function without `sign` or `abs`. No FFT, no 5-D reshape.

In [ ]:
%%writefile safawt_backbone.py
"""Paper 73 backbone for YOLO26n: SAFA-WT (Haar wavelet + soft-threshold + frequency attention) inside the C3k2 at layers 6 and 8."""
import torch
import torch.nn as nn
import torch.nn.functional as F

from ultralytics.models.yolo.detect import DetectionTrainer
from ultralytics.nn.modules.block import C3k2
from ultralytics.nn.tasks import DetectionModel
from ultralytics.utils import RANK

SAFA_LAYERS = {6: 2, 8: 1}  # layer index: number of wavelet levels


def haar_filters(c):
    """The 4 orthonormal 2x2 Haar filters (LL, LH, HL, HH), each repeated for c depthwise channels: (4, c, 1, 2, 2)."""
    f = 0.5 * torch.tensor([[[1, 1], [1, 1]], [[1, -1], [1, -1]], [[1, 1], [-1, -1]], [[1, -1], [-1, 1]]], dtype=torch.float32)
    return f[:, None, None].repeat(1, c, 1, 1, 1)


class SoftThreshold(nn.Module):
    """sign(y) * max(|y| - tau, 0) written as y - min(max(y, -tau), tau); one learnable tau per channel."""

    def __init__(self, c, init=0.02):
        super().__init__()
        self.t = nn.Parameter(torch.full((c,), init))  # 1-D, so MuSGD treats it as a plain SGD parameter

    def forward(self, y):
        tau = self.t.abs().view(1, -1, 1, 1)
        return y - torch.minimum(torch.maximum(y, -tau), tau)


class FreqAttention(nn.Module):
    """GAP over the 4 concatenated bands -> 1x1 conv -> ReLU -> 1x1 conv -> sigmoid: one weight per band and channel."""

    def __init__(self, c, r=16):
        super().__init__()
        h = max(4 * c // r, 8)
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Sequential(nn.Conv2d(4 * c, h, 1), nn.ReLU(inplace=True), nn.Conv2d(h, 4 * c, 1), nn.Sigmoid())

    def forward(self, x):
        return self.fc(self.pool(x))


class SAFAWT(nn.Module):
    """SAFA-WT: L-level Haar DWT, depthwise 3x3 per band, ST on high bands, FA weights, weighted IWT (deepest level first)."""

    def __init__(self, c, levels=1):
        super().__init__()
        self.c, self.levels = c, levels
        self.register_buffer("wt", haar_filters(c))  # fixed, frozen, saved with the model
        self.band_conv = nn.ModuleList(nn.ModuleList(nn.Conv2d(c, c, 3, 1, 1, groups=c) for _ in range(4)) for _ in range(levels))
        self.st = nn.ModuleList(nn.ModuleList(SoftThreshold(c) for _ in range(3)) for _ in range(levels))
        self.fa = nn.ModuleList(FreqAttention(c) for _ in range(levels))

    def dwt(self, x):
        return [F.conv2d(x, self.wt[b], stride=2, groups=self.c) for b in range(4)]

    def iwt(self, bands):
        return sum(F.conv_transpose2d(y, self.wt[b], stride=2, groups=self.c) for b, y in enumerate(bands))

    def forward(self, x):
        c = self.c
        sizes, levels, cur = [], [], x
        for i in range(self.levels):  # forward decomposition
            h, w = cur.shape[2:]
            sizes.append((h, w))
            if h % 2 or w % 2:  # odd size (e.g. rect validation): pad to even, crop after IWT
                cur = F.pad(cur, (0, w % 2, 0, h % 2), mode="replicate")
            bands = self.dwt(cur)
            levels.append([conv(y) for conv, y in zip(self.band_conv[i], bands)])
            cur = bands[0]  # the next level splits the LL band again
        z = None
        for i in reversed(range(self.levels)):  # adaptive reconstruction, deepest level first
            y_ll, *y_high = levels[i]
            if z is not None:
                y_ll = y_ll + z
            y_high = [st(y) for st, y in zip(self.st[i], y_high)]
            bands = [y_ll] + y_high
            a = self.fa[i](torch.cat(bands, 1))
            z = self.iwt([a[:, b * c:(b + 1) * c] * y for b, y in enumerate(bands)])
            h, w = sizes[i]
            z = z[:, :, :h, :w]
        return z


class C3kSAFA(nn.Module):
    """A pretrained C3k block followed by a residual SAFA-WT: y + gamma * SAFA-WT(y). gamma starts at 0 (identity start)."""

    def __init__(self, block, c, levels=1):
        super().__init__()
        self.block = block
        self.safa = SAFAWT(c, levels)
        self.gamma = nn.Parameter(torch.zeros(c))  # 1-D, so MuSGD treats it as a plain SGD parameter

    def forward(self, x):
        y = self.block(x)
        return y + self.gamma.view(1, -1, 1, 1) * self.safa(y)


def add_safawt(model, layers=SAFA_LAYERS):
    """Put SAFA-WT after the C3k inside the C3k2 at the given layers (all pretrained weights kept). Returns the model."""
    for i, lv in layers.items():
        blk = model.model[i]
        assert isinstance(blk, C3k2), f"layer {i} is {type(blk).__name__}, not C3k2"
        for j, inner in enumerate(blk.m):
            if isinstance(inner, C3kSAFA):
                continue  # already added
            blk.m[j] = C3kSAFA(inner, blk.c, lv).to(next(inner.parameters()).device)
        blk.np = sum(p.numel() for p in blk.parameters())  # C3k2 keeps its i, f, type routing attributes
    return model


class SAFAWTTrainer(DetectionTrainer):
    """Loads the pretrained weights first, then adds SAFA-WT inside the C3k2 at layers 6 and 8 (gamma = 0 start)."""

    def get_model(self, cfg=None, weights=None, verbose=True):
        model = DetectionModel(cfg, nc=self.data["nc"], ch=self.data["channels"], verbose=False)
        if weights:
            model.load(weights)
        add_safawt(model)
        if RANK in {-1, 0}:
            print(f"Paper 73 backbone: SAFA-WT in the C3k2 at layers {list(SAFA_LAYERS)} (levels {list(SAFA_LAYERS.values())})")
        if verbose and RANK in {-1, 0}:
            model.info()
        return model


In [ ]:
import importlib, safawt_backbone
importlib.reload(safawt_backbone)
import torch.nn.functional as F
from safawt_backbone import add_safawt, SAFAWT, SAFAWTTrainer, SAFA_LAYERS
from ultralytics.nn.tasks import DetectionModel, yaml_model_load
from ultralytics.utils.torch_utils import get_flops

def fused_params_gflops(model, imgsz=640):
    m = deepcopy(model).float().cpu().eval()
    m.fuse(verbose=False)
    return sum(p.numel() for p in m.parameters()) / 1e6, get_flops(m, imgsz)

# 1) Haar DWT -> IWT must rebuild the input exactly (also for an odd size)
s = SAFAWT(8, 1)
for hw in ((40, 40), (15, 21)):
    t = torch.randn(1, 8, *hw)
    tp = F.pad(t, (0, hw[1] % 2, 0, hw[0] % 2), mode="replicate")
    print(f"DWT -> IWT {hw}: max diff {(s.iwt(s.dwt(tp))[:, :, :hw[0], :hw[1]] - t).abs().max():.2e} (should be about 1e-7)")

# 2) structure check (untrained): base vs Paper 73 backbone; gamma = 0, so outputs must match the base exactly
base = DetectionModel(yaml_model_load("yolo26n.yaml"), nc=len(names), verbose=False).eval()
safa = add_safawt(deepcopy(base)).eval()
with torch.no_grad():
    xb = xs = torch.randn(1, 3, 640, 640)
    for i in range(9):  # backbone layers 0-8 are sequential (from = -1)
        xb, xs = base.model[i](xb), safa.model[i](xs)
        if i in SAFA_LAYERS:
            print(f"layer {i}: output {tuple(xs.shape[1:])} | diff to base {(xb - xs).abs().max():.2e} (should be 0 at gamma = 0)")
    _ = safa(torch.zeros(1, 3, 640, 640))  # full forward check
    _ = safa(torch.zeros(1, 3, 640, 352))  # rect-style input: P5 is 11 wide (odd) -> padding path
print("SAFA-WT weights:", sum(p.numel() for i in SAFA_LAYERS for n, p in safa.model[i].named_parameters() if ".safa." in n or "gamma" in n))
for tag, m in (("base", base), ("Paper 73 backbone", safa)):
    pm, gf = fused_params_gflops(m)
    print(f"{tag:18s} Params {pm:.4f} M | GFLOPs {gf:.3f}")

## 4. Training

Only the Paper 73 backbone model is trained.

In [ ]:
from ultralytics import YOLO

EPOCHS  = 100
IMGSZ   = 640
BATCH   = 16
WORKERS = 2
SEED    = 0
PROJECT = str(WORK / "runs")

COMMON = dict(
    data=DATA_YAML, epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH, seed=SEED, deterministic=True,
    pretrained="yolo26n.pt", device=0, workers=WORKERS, cache=False, patience=30, cos_lr=False,
    project=PROJECT, exist_ok=True, plots=True,
    # channels_last=False: ultralytics' MuSGD crashes on channels_last weights ("view size is not compatible").
    optimizer="MuSGD", lr0=0.01, momentum=0.9, channels_last=False,
)

# (row label in the table, run name)
VARIANTS = [
    ("SAFA-WT Backbone (Paper 73)", "bb_safawt"),
]

def best_path(run):
    return Path(PROJECT) / f"{run}_s{SEED}" / "weights" / "best.pt"

for label, run in VARIANTS:
    print(f"\n========== {label} ({run}) ==========")
    model = YOLO("yolo26n.yaml")
    model.train(trainer=SAFAWTTrainer, **dict(COMMON, name=f"{run}_s{SEED}"))
    del model; gc.collect(); torch.cuda.empty_cache()

## 5. Test-set evaluation: Precision, Recall, F1-score, mAP@50, Params, GFLOPs

Params and GFLOPs are counted on the fused inference model, the same way as the baseline row.

In [ ]:
import safawt_backbone  # must be imported before loading a SAFA-WT checkpoint

rows = []
for label, run in VARIANTS:
    w = best_path(run)
    if not w.exists():
        print(f"{run}: best.pt not found, train it first"); continue
    r = YOLO(str(w)).val(data=DATA_YAML, split="test", imgsz=IMGSZ, batch=16, device=0, plots=False, verbose=False)
    P, R = float(r.box.mp), float(r.box.mr)
    params, gflops = fused_params_gflops(YOLO(str(w)).model, IMGSZ)
    rows.append({"run": run, "Backbone Variant": label,
                 "Params (M)": round(params, 2), "GFLOPs": round(gflops, 2),
                 "Precision": round(P, 3), "Recall": round(R, 3),
                 "F1-score": round(2 * P * R / (P + R + 1e-9), 3),
                 "mAP@50": round(float(r.box.map50), 3)})
METRICS = pd.DataFrame(rows)
METRICS.drop(columns="run")

## 6. Package for Raspberry Pi 5 and Jetson

Creates `paper73_backbone_edge_bench.zip` in the **Output** tab with:

| Item | Used on |
|---|---|
| `bb_safawt_ncnn_model/` (NCNN model, imgsz 640) | Raspberry Pi 5 |
| `bb_safawt.pt` | Jetson |
| `bench_images/` (50 real test images, seed 0) | both |
| `safawt_backbone.py` | Jetson (needed to load the SAFA-WT `.pt`) |
| `pi5_bench.py`, `jetson_bench.py` | benchmark scripts |

Both scripts: batch 1, imgsz 640, cycle through the 50 real test images (loaded into memory first),
100 warm-up runs, then 1000 timed runs end-to-end (preprocessing + inference + postprocessing), and print the **median** ms and FPS.
Unzip into its own folder, e.g. `python -m zipfile -e paper73_backbone_edge_bench.zip paper73_backbone_edge_bench`.

In [ ]:
EDGE_IMGSZ = 640
N_BENCH_IMAGES = 50
EDGE = WORK / "paper73_backbone_edge_bench"
shutil.rmtree(EDGE, ignore_errors=True)
EDGE.mkdir()
shutil.copy("safawt_backbone.py", EDGE / "safawt_backbone.py")

# 50 real test images (fixed seed, so every model and device uses the same images)
test_imgs = sorted(p for p in (DATA_ROOT / "test" / "images").glob("*") if p.suffix.lower() in {".jpg", ".jpeg", ".png", ".bmp"})
random.Random(SEED).shuffle(test_imgs)
(EDGE / "bench_images").mkdir()
for p in test_imgs[:N_BENCH_IMAGES]:
    shutil.copy(p, EDGE / "bench_images" / p.name)
print(f"bench_images: {len(list((EDGE / 'bench_images').glob('*')))} real test images")

for _, run in VARIANTS:
    w = best_path(run)
    if not w.exists():
        print(f"{run}: best.pt not found, skipped"); continue
    shutil.copy(w, EDGE / f"{run}.pt")
    p = YOLO(str(w)).export(format="ncnn", imgsz=EDGE_IMGSZ, half=False)
    shutil.copytree(p, EDGE / f"{run}_ncnn_model")  # ultralytics needs the "_ncnn_model" suffix
    print(f"{run:12s} -> {run}.pt, {run}_ncnn_model/")

COMMON_BENCH = """
import glob, statistics, sys, time
import cv2

IMGSZ, WARMUP, RUNS = {imgsz}, 100, 1000
imgs = [cv2.imread(p) for p in sorted(glob.glob("bench_images/*"))]
imgs = [im for im in imgs if im is not None]
if not imgs:
    sys.exit("No images found in bench_images/. Run this script inside the unzipped folder.")
print(f"{{len(imgs)}} real test images, imgsz {{IMGSZ}}, batch 1, {{WARMUP}} warm-up + {{RUNS}} timed runs (median)")


def bench(model, **kw):
    for i in range(WARMUP):
        model.predict(imgs[i % len(imgs)], imgsz=IMGSZ, verbose=False, **kw)
    times = []
    for i in range(RUNS):
        t = time.perf_counter()
        model.predict(imgs[i % len(imgs)], imgsz=IMGSZ, verbose=False, **kw)
        times.append((time.perf_counter() - t) * 1000)
    return statistics.median(times)
""".format(imgsz=EDGE_IMGSZ)

PI5 = '''# Raspberry Pi 5 benchmark (NCNN). Run inside the unzipped folder:  python pi5_bench.py
from ultralytics import YOLO
''' + COMMON_BENCH + '''
models = sorted(glob.glob("*_ncnn_model"))
if not models:
    sys.exit("No *_ncnn_model folders found.")
for path in models:
    ms = bench(YOLO(path, task="detect"))
    print(f"{path.replace('_ncnn_model', ''):12s}  median {ms:8.2f} ms   {1000 / ms:7.2f} FPS")
'''

JETSON = '''# Jetson benchmark (PyTorch, CUDA FP16). Run inside the unzipped folder:  python jetson_bench.py
import sys
import torch

if not torch.cuda.is_available():
    sys.exit("WARNING: CUDA GPU not available. Install the NVIDIA Jetson PyTorch wheel for your JetPack, then run again.")
print("GPU:", torch.cuda.get_device_name(0))

sys.path.insert(0, ".")
import safawt_backbone  # needed to load the SAFA-WT checkpoint
from ultralytics import YOLO
''' + COMMON_BENCH + '''
models = sorted(glob.glob("*.pt"))
if not models:
    sys.exit("No .pt files found.")
for path in models:
    ms = bench(YOLO(path, task="detect"), device=0, half=True)
    print(f"{path.replace('.pt', ''):12s}  median {ms:8.2f} ms   {1000 / ms:7.2f} FPS")
'''

(EDGE / "pi5_bench.py").write_text(PI5)
(EDGE / "jetson_bench.py").write_text(JETSON)
shutil.make_archive(str(WORK / "paper73_backbone_edge_bench"), "zip", EDGE)
print("\nDownload paper73_backbone_edge_bench.zip from the Output tab.")

### How to run on the devices

**Raspberry Pi 5** (same Pi and same conditions as the baseline measurement)
1. Activate the Python environment that has `ultralytics==8.4.137` and `ncnn`.
2. `echo performance | sudo tee /sys/devices/system/cpu/cpu*/scaling_governor`
3. Go into the unzipped folder and run `python pi5_bench.py`.
4. Run it 3 times with a 2–3 minute pause; use the middle value. Check `vcgencmd measure_temp` and `vcgencmd get_throttled` (`throttled=0x0` = OK).

**Jetson**
1. NVIDIA Jetson PyTorch + torchvision for your JetPack, then `ultralytics==8.4.137`.
2. `sudo nvpmodel -m 0` (check with `sudo nvpmodel -q`) and `sudo jetson_clocks`.
3. Go into the unzipped folder and run `python jetson_bench.py`. Run 3 times; use the middle value.

## 7. Final backbone table

The baseline row is already filled in. Type the median FPS of the Paper 73 backbone model
(and the baseline Jetson FPS when you have it), keep `None` for a missing value (shown as `--`), then run the cell.
Run section 5 first in the same session (it fills `METRICS`).

In [ ]:
# baseline values (identical YOLO26n model)
BASE_ROW = {"Backbone Variant": "Standard YOLO26n Backbone", "Params (M)": 2.38, "GFLOPs": 5.34,
            "Precision": 0.971, "Recall": 0.937, "F1-score": 0.953, "mAP@50": 0.958,
            "FPS (Rasp-5)": 13.38, "FPS (Jetson)": None}

FPS_PI5    = {"bb_safawt": None}
FPS_JETSON = {"bb_safawt": None}

table = METRICS.copy()
table["FPS (Rasp-5)"] = table["run"].map(lambda r: FPS_PI5.get(r))
table["FPS (Jetson)"] = table["run"].map(lambda r: FPS_JETSON.get(r))
COLS = ["Backbone Variant", "Params (M)", "GFLOPs", "Precision", "Recall", "F1-score", "mAP@50", "FPS (Rasp-5)", "FPS (Jetson)"]
table = pd.concat([pd.DataFrame([BASE_ROW]), table[COLS]], ignore_index=True)[COLS]
table = table.astype(object).where(table.notna(), "--")
table.style.hide(axis="index")

## 8. Zip all results

Run this last. It creates **`paper73_backbone_all_results.zip`** in the **Output** tab with:

| Folder in the zip | Contents |
|---|---|
| `paper73_backbone_edge_bench/` | NCNN + `.pt` model, 50 bench images, `safawt_backbone.py`, `pi5_bench.py`, `jetson_bench.py` |
| `weights/` | `best.pt`, renamed `bb_safawt_best.pt` |
| `runs/` | full training folder: `best.pt`, `last.pt`, `results.csv`, training curves, confusion matrix, PR curves |

In [ ]:
import zipfile

ALL_ZIP = WORK / "paper73_backbone_all_results.zip"
folders = [WORK / "paper73_backbone_edge_bench", WORK / "runs"]

with zipfile.ZipFile(ALL_ZIP, "w", zipfile.ZIP_DEFLATED) as z:
    for folder in folders:
        if not folder.exists():
            print(f"{folder.name}: not found, skipped"); continue
        n = 0
        for f in folder.rglob("*"):
            if f.is_file():
                z.write(f, f.relative_to(WORK))
                n += 1
        print(f"{folder.name + '/':30s} {n} files")
    for _, run in VARIANTS:
        w = best_path(run)
        if w.exists():
            z.write(w, f"weights/{run}_best.pt")
            print(f"weights/{run}_best.pt")

print(f"\nDownload {ALL_ZIP.name} ({ALL_ZIP.stat().st_size / 1e6:.1f} MB) from the Output tab.")